# Visión Artificial en Tiempo Real: Clasificación de Botellas y Tazas

## Descripción General
Este cuaderno implementa un sistema de visión artificial puro para ejecutar en tiempo real utilizando la cámara web. Utilizaremos el modelo YOLOv8 para identificar específicamente dos clases de objetos comunes: botellas y tazas/vasos.

A diferencia de la versión completa, este script **no requiere conexión a hardware externo** (microcontroladores o servomotores). Su objetivo es puramente visual: abrir el flujo de video, procesarlo con la IA y mostrar los resultados de clasificación y localización inmediatamente en la pantalla.

## Requisitos de Software
1. **Python 3.8+**
2. **Ultralytics** (para YOLOv8)
3. **OpenCV** (para manipulación de video)

In [1]:
# ==============================================================================
# 1. INSTALACIÓN E IMPORTACIÓN DE LIBRERÍAS (PURO SOFTWARE)
# ==============================================================================
# Descomenta la siguiente línea si necesitas instalar las librerías necesarias:
# !pip install ultralytics opencv-python pyttsx3

import pyttsx3
import cv2
from ultralytics import YOLO

print("Librerías importadas correctamente. Entorno listo para la detección visual.")

Librerías importadas correctamente. Entorno listo para la detección visual.


## 2. Carga del Modelo y Definición de Filtros
Cargaremos el modelo preentrenado `yolov8n.pt` (Nano), que es el más rápido y eficiente para ejecutar en CPUs de laptops sin necesidad de tarjetas gráficas potentes.

Para más información consulta: https://docs.ultralytics.com/es/models/yolov8

**Habilidad clave:** YOLOv8 preentrenado en COCO detecta 80 clases. Para este ejercicio, necesitamos filtrar los resultados para quedarnos solo con:
*   ID 39: `bottle` (botella)
*   ID 41: `cup` (taza/vaso)

Consulta https://docs.ultralytics.com/es/datasets/detect/coco para conocer los IDs de las clases.

In [2]:
# 2.1 Cargar el modelo YOLO Nano preentrenado (aprox. 6 MB)
# Si no existe en la carpeta, se descargará automáticamente.
modelo = YOLO("yolov8n.pt") 

# 2.2 Definir los IDs de las clases que nos interesan del dataset COCO.
# Bottle (Botella) es el índice 39.
# Cup (Taza/Vaso/Lata) es el índice 41.
# Creamos una lista para pasarla como argumento al predictor.
# Classes
# names:
#   0: person
#   39: bottle
#   41: cup
#   67: cell phone
clases_a_detectar = [0, 39, 41, 67]

print(f"Modelo cargado. Filtrando para detectar solo las clases COCO IDs: {clases_a_detectar}")

Modelo cargado. Filtrando para detectar solo las clases COCO IDs: [0, 39, 41, 67]


## 3. Ejecución del Bucle Principal en Tiempo Real (OpenCV)
En esta etapa, abrimos la cámara web, capturamos frames continuamente, los enviamos a YOLO aplicando el filtro de clases definido anteriormente, y dibujamos los resultados directamente sobre el video que se muestra al usuario.

In [ ]:
# ==============================================================================
# 3. BUCLE DE CLASIFICACIÓN EN TIEMPO REAL
# ==============================================================================

# 3.1 Inicializar la cámara web
cap = cv2.VideoCapture(0)

# Inicializamos motor de voz
motor_voz = pyttsx3.init()

if not cap.isOpened():
    print("Error: No se pudo acceder a la cámara web.")
else:
    print("Cámara iniciada. Presiona la tecla 'q' dentro de la ventana de video para salir.")

UMBRAL_CONFIANZA = 0.5  # Mostrar detecciones con >50% de certeza

while cap.isOpened():
    # A. Capturar frame por frame
    exito, frame = cap.read()
    
    if not exito:
        print("Error al leer el frame de la cámara.")
        break

    # B. Inferencia Directa
    # Al quitar stream=True y pasar un solo frame, YOLO devuelve una lista con 1 resultado.
    # Usamos modelo() directamente, que es la forma recomendada en versiones recientes.
    resultados = modelo(frame, conf=UMBRAL_CONFIANZA, classes=clases_a_detectar, verbose=False)
    
    # Extraer el resultado de la imagen actual
    resultado_actual = resultados[0]
    
    # C. Procesamiento y Visualización
    # .plot() genera automáticamente la imagen con las cajas de detección dibujadas
    frame_con_detecciones = resultado_actual.plot()
    
    # Imprimir en consola solo si detecta algo
    for box in resultado_actual.boxes:
        cls_id = int(box.cls[0])
        nom_clase = modelo.names[cls_id]
        print(f"Detectado: {nom_clase} (Conf: {box.conf[0]:.2f})")

        if nom_clase in ["person"]:
            motor_voz.say("Detectado persona")
            
        #motor_voz.runAndWait()  # Ejecutar el motor de voz para cada detección

    # D. Mostrar el frame resultante en una ventana
    cv2.imshow("Clasificador Tiempo Real: Bottle & Cup", frame_con_detecciones)

    # E. Condición de salida: Presionar la tecla 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Cerrando el sistema por solicitud del usuario.")
        break

# ==============================================================================
# 4. LIMPIEZA DE RECURSOS
# ==============================================================================
cap.release()
cv2.destroyAllWindows()
motor_voz.stop()  # Detener cualquier reproducción de voz en curso
print("Recursos de video liberados correctamente.")

Cámara iniciada. Presiona la tecla 'q' dentro de la ventana de video para salir.
Detectado: person (Conf: 0.83)
Detectado: person (Conf: 0.84)
Detectado: person (Conf: 0.85)
Detectado: person (Conf: 0.83)
Detectado: person (Conf: 0.82)
Detectado: person (Conf: 0.84)
Detectado: person (Conf: 0.85)
Detectado: person (Conf: 0.86)
Detectado: person (Conf: 0.87)
Detectado: person (Conf: 0.84)
Detectado: person (Conf: 0.85)
Detectado: person (Conf: 0.86)
Detectado: person (Conf: 0.86)
Detectado: person (Conf: 0.85)
Detectado: person (Conf: 0.85)
Detectado: person (Conf: 0.86)
Detectado: person (Conf: 0.86)
Detectado: person (Conf: 0.86)
Detectado: person (Conf: 0.86)
Detectado: person (Conf: 0.86)
Detectado: person (Conf: 0.86)
Detectado: person (Conf: 0.86)
Detectado: person (Conf: 0.86)
Detectado: person (Conf: 0.86)
Detectado: person (Conf: 0.86)
Detectado: person (Conf: 0.86)
Detectado: person (Conf: 0.86)
Detectado: person (Conf: 0.86)
Detectado: person (Conf: 0.86)
Detectado: person (C

In [4]:
# # Iniciamos el motor de voz
# motor_voz = pyttsx3.init()

# # Indicamos el texto a reproducir
# motor_voz.say("Hola Bloque")

# # Ejecutamos el motor de voz
# motor_voz.runAndWait()

In [5]:
# rate = motor_voz.getProperty('rate')
# print(f"Velocidad de voz: {rate}")

In [6]:
# motor_voz.setProperty('rate', 150) # Ajusta la velocidad de voz a 150 palabras por minuto

In [7]:
# voces = motor_voz.getProperty('voices')

# for voz in voces:
#     print(f"ID: {voz.id}, Nombre: {voz.name}, Idioma: {voz.languages}")

In [8]:
# motor_voz.setProperty('voice', voces[1].id) # 1 para feminina, 0 para masculina (según disponibilidad en el sistema operativo)

In [9]:
# # Indicamos el texto a reproducir
# motor_voz.say("Hola Bloque")

# # Ejecutamos el motor de voz
# motor_voz.runAndWait()